In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda

In [3]:
# Suppress all UserWarnings coming specifically from the pydantic module and its submodules

import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="pydantic.*"
)

## Logical Routing

![Logical Routing Image](Images/Logical_Routing.png)

### Logical Routing is the process where an LLM directs a user query to the most relevant data source based on explicit criteria, such as the programming language mentioned in the question.

![LLM with Structured Output Image](Images/LLM_WithStructuredOutput.png)

In [4]:
# Define the Data Model 

class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question, choose which datasource is most relevant.",
    )

In [5]:
# LLM with function call 

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)

In [6]:
# Create the Prompt

system_instruction = """You are an expert at routing a user question to the appropriate data source.
Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    ("human", "{question}"),
])

In [7]:
# Build the First Half: The Router Chain. This evaluates the question and outputs a RouteQuery object.
router = prompt | structured_llm

In [8]:
# Define the Execution Logic (The Switchboard)

def choose_route(routing_decision: RouteQuery):
    """Takes the LLM's decision and triggers the actual downstream chains."""
    
    # In a real app, you would return actual LangChain retrievers/chains here
    if routing_decision.datasource == "python_docs":
        return "Route chosen: Python. --> Executing Python RAG Chain..."
        
    elif routing_decision.datasource == "js_docs":
        return "Route chosen: JavaScript. --> Executing JS RAG Chain..."
        
    elif routing_decision.datasource == "golang_docs":
        return "Route chosen: Golang. --> Executing Go RAG Chain..."
        
    else:
        return "Error: Unknown datasource."

In [9]:
# Assemble the Full Pipeline, Connect the router's output directly into the execution logic

full_chain = router | RunnableLambda(choose_route)

In [10]:
# Test the Pipeline
test_question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": test_question})

In [11]:
result

RouteQuery(datasource='python_docs')

In [12]:
type(result)

__main__.RouteQuery

In [13]:
result.datasource

'python_docs'

In [14]:
print(full_chain.invoke({"question": test_question}))

Route chosen: Python. --> Executing Python RAG Chain...


## Semantic Routing

![Semantic Routing Image](Images/Semantic_Routing.png)

### Semantic Routing is the process of directing a user query to the most relevant data source by comparing the semantic similarity between the query and available options using embeddings.

In [15]:
import numpy as np
from typing import Dict
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [16]:
# Define the Route Prompts
PHYSICS_TEMPLATE = """You are a very smart physics professor.
You are great at answering questions about physics in a concise and easy to understand manner.
When you don't know the answer to a question you admit that you don't know.
Write your answer in plain text, without LaTeX.

Here is a question:
{query}"""

MATH_TEMPLATE = """You are a very good mathematician. You are great at answering math questions.
You are so good because you are able to break down hard problems into their component parts,
answer the component parts, and then put them together to answer the broader question.
Write your answer in plain text, without LaTeX.

Here is a question:
{query}"""

In [17]:
# Initialize Models

embeddings_model = OpenAIEmbeddings(model = "text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [18]:
# Pre-compute Route Embeddings

prompt_templates = [PHYSICS_TEMPLATE, MATH_TEMPLATE]
print("\n--- Initialization ---")
print("Embedding the Physics and Math templates...")
prompt_embeddings = embeddings_model.embed_documents(prompt_templates)
print(f"Type of prompt_embeddings: {type(prompt_embeddings)}")
print(f"Number of embeddings generated: {len(prompt_embeddings)}")
print(f"Length of a single embedding vector (Dimensions): {len(prompt_embeddings[0])}")


--- Initialization ---
Embedding the Physics and Math templates...
Type of prompt_embeddings: <class 'list'>
Number of embeddings generated: 2
Length of a single embedding vector (Dimensions): 1536


In [19]:
# Define the Execution Logic (The Vector Switchboard)

def prompt_router(input_dict: Dict[str, str]) -> PromptTemplate:
    """Embeds the user query, calculates similarity to routes, and returns the best prompt."""
    
    # Check the incoming input
    print(f"1. Raw Input received by router:\n   Value: {input_dict}\n   Type: {type(input_dict)}")
    
    query = input_dict["query"]
    
    # Step A: Turn the user query into a vector
    query_embedding = embeddings_model.embed_query(query)
    print(f"\n2. Query Embedding generated:")
    print(f"   Type: {type(query_embedding)}")
    print(f"   Length: {len(query_embedding)} dimensions")
    print(f"   First 3 values: {query_embedding[:3]}...")
    
    # Step B: Compute cosine similarity between the query and all templates
    raw_similarity_matrix = cosine_similarity([query_embedding], prompt_embeddings)
    similarity_scores = raw_similarity_matrix[0] 
    
    print(f"\n3. Cosine Similarity Calculation:")
    print(f"Raw Matrix:\n{raw_similarity_matrix}")
    print(f"   Raw Matrix Type: {type(raw_similarity_matrix)}")
    print(f"   Scores Array Type: {type(similarity_scores)}")
    print(f"   Scores [Physics, Math]: {similarity_scores}")
    
    # Step C: Find the index of the highest score
    best_match_index = np.argmax(similarity_scores)
    most_similar_prompt = prompt_templates[best_match_index]
    
    route_name = "MATH" if most_similar_prompt == MATH_TEMPLATE else "PHYSICS"
    print(f"\n4. Routing Decision:")
    print(f"   Winning Index: {best_match_index}")
    print(f"   Chosen Route: {route_name}")
    
    # Return the PromptTemplate object
    final_prompt = PromptTemplate.from_template(most_similar_prompt)
    print(f"\n5. Output of Router:\n   Type: {type(final_prompt)}")
    
    return final_prompt

In [20]:
# Assemble the Full Pipeline (LCEL)

chain = (
    {"query": RunnablePassthrough()}  
    | RunnableLambda(prompt_router)   
    | llm                             
    | StrOutputParser()             
)

In [21]:
# Test the Pipeline

print("\nTesting Physics Query:")
final_answer = chain.invoke("What happens to time near a black hole?")
print(f"Type of final output: {type(final_answer)}")
print(f"Answer:\n{final_answer}")

print("\nTesting Math Query:")
final_answer = chain.invoke("What is the integral of x squared?")
print(f"Type of final output: {type(final_answer)}")
print(f"Answer:\n{final_answer}")


Testing Physics Query:
1. Raw Input received by router:
   Value: {'query': 'What happens to time near a black hole?'}
   Type: <class 'dict'>

2. Query Embedding generated:
   Type: <class 'list'>
   Length: 1536 dimensions
   First 3 values: [-0.026214599609375, 0.01261138916015625, 0.000560760498046875]...

3. Cosine Similarity Calculation:
Raw Matrix:
[[0.16359219 0.12570839]]
   Raw Matrix Type: <class 'numpy.ndarray'>
   Scores Array Type: <class 'numpy.ndarray'>
   Scores [Physics, Math]: [0.16359219 0.12570839]

4. Routing Decision:
   Winning Index: 0
   Chosen Route: PHYSICS

5. Output of Router:
   Type: <class 'langchain_core.prompts.prompt.PromptTemplate'>
Type of final output: <class 'langchain_core.messages.base.TextAccessor'>
Answer:
Near a black hole, time behaves differently due to the effects of gravity on spacetime, a concept described by Einstein's theory of general relativity. As you get closer to a black hole, the gravitational pull becomes extremely strong, whi

## Query Construction

![Query Construction Image](Images/Query_Construction.png)

### Query Construction is the process of transforming a user's question into a structured database query, often using an LLM to extract relevant fields and filters for optimized retrieval from a specific dataset.

In [22]:
import datetime
from typing import Optional
from pydantic import BaseModel, Field
from langchain_community.document_loaders import YoutubeLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [23]:
# Load Documents and Metadata

print("--- 1. Loading YouTube Video ---")

docs = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=VW6LYuli7VU", 
    add_video_info=False 
).load()

docs[0].metadata = {
    'source': 'VW6LYuli7VU', 
    'title': 'Richard Feynman talks about Algebra', 
    'description': 'Unknown', 
    'view_count': 1885729, 
    'thumbnail_url': 'https://i.ytimg.com/vi/VW6LYuli7VU/hq720.jpg', 
    'publish_date': '2014-01-22 00:00:00', 
    'length': 82, 
    'author': 'David Petro'
}

# Let's inspect what we have now
video_metadata = docs[0].metadata
print(f"Type of metadata: {type(video_metadata)}\n")
print(f"Metadata content: {video_metadata}\n")

--- 1. Loading YouTube Video ---
Type of metadata: <class 'dict'>

Metadata content: {'source': 'VW6LYuli7VU', 'title': 'Richard Feynman talks about Algebra', 'description': 'Unknown', 'view_count': 1885729, 'thumbnail_url': 'https://i.ytimg.com/vi/VW6LYuli7VU/hq720.jpg', 'publish_date': '2014-01-22 00:00:00', 'length': 82, 'author': 'David Petro'}



##### How a question becomes a structured query

**The key point:** The LLM searches nothing here. It fills in a form, the `TutorialSearch` schema: Two search strings, plus filters it should fill only when the question asks for them. A database would then run the filled form as its query.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
"richard feynman videos with over 1 million views"
     │  query_analyzer = prompt | llm.with_structured_output(TutorialSearch)
     ▼
content_search = "richard feynman"   ──► searched in the transcripts
title_search   = "richard feynman"   ──► searched in the titles
min_view_count = 1000000             ──► filter: view_count &gt;= 1000000
every other filter stays None
</pre>

- **The video above:** Shows the metadata each video in such a database would have (`view_count`, `publish_date`, `length`). That's why the schema has those filters. This notebook only builds the query and never searches the video.
- **`Field(description=...)`:** The description is sent to the model with the schema, so it works like an instruction for that one field. A field with no default (`content_search`, `title_search`) must be filled. `Field(None, ...)` gives the field a default of `None`, so it may stay empty.
- **The wording matters:** The descriptions are prompt text. When they were shortened in a test, the model left `content_search` empty for two of the four questions, so the original wording is kept.
- **"Only use if explicitly specified" is a request, not a rule:** For "published before 2015" the model still adds an earliest date that nobody asked for (see the second question below). Descriptions guide the model, but they can't force it.
- **`with_structured_output(TutorialSearch)`:** Makes the model reply with a filled-in `TutorialSearch` object instead of text.
- **`model_dump_json(indent=2, exclude_none=True)`:** Prints the filled form as JSON and leaves out the filters that stayed `None`.

In [24]:
# Define the Structured Query Schema

print("--- 2. Defining Pydantic Schema ---")

class TutorialSearch(BaseModel):
    """Search over a database of historic math and science lectures."""

    content_search: str = Field(description="Similarity search query applied to video transcripts.")
    title_search: str = Field(description="Alternate version of the content search query to apply to video titles. Should be succinct and only include key words that could be in a video title.")

    min_view_count: Optional[int] = Field(None, description="Minimum view count filter, inclusive. Only use if explicitly specified.")
    max_view_count: Optional[int] = Field(None, description="Maximum view count filter, exclusive. Only use if explicitly specified.")
    earliest_publish_date: Optional[datetime.date] = Field(None, description="Earliest publish date filter, inclusive. Only use if explicitly specified.")
    latest_publish_date: Optional[datetime.date] = Field(None, description="Latest publish date filter, exclusive. Only use if explicitly specified.")
    min_length_sec: Optional[int] = Field(None, description="Minimum video length in seconds, inclusive. Only use if explicitly specified.")
    max_length_sec: Optional[int] = Field(None, description="Maximum video length in seconds, exclusive. Only use if explicitly specified.")

--- 2. Defining Pydantic Schema ---


In [25]:
# Set up the LLM and the Pipeline 

print("--- 3. Setting up the LLM and Chain ---")

system_instruction = """You are an expert at converting user questions into database queries. \
You have access to a database of historic math and physics video lectures. \
Given a question, return a database query optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    ("human", "{question}"),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# This is the magic step: We force the LLM to output our Pydantic schema
structured_llm = llm.with_structured_output(TutorialSearch)

# Create the pipeline (Prompt -> LLM enforces structure -> Output)
query_analyzer = prompt | structured_llm

print(f"Type of query_analyzer: {type(query_analyzer)}\n")

--- 3. Setting up the LLM and Chain ---
Type of query_analyzer: <class 'langchain_core.runnables.base.RunnableSequence'>



In [26]:
# Testing the Pipeline 

print("--- 4. Testing the Query Analyzer ---")

test_questions = [
    "feynman talking about algebra",
    "lectures on math published before 2015",
    "short videos under 3 minutes explaining variables like x and y",
    "richard feynman videos with over 1 million views"
]

for q in test_questions:
    print(f"User Question: '{q}'")
    
    result = query_analyzer.invoke({"question": q})
    
    print(f"   Type of raw result: {type(result)}")
    
    print(result.model_dump_json(indent=2, exclude_none=True))
    print()

--- 4. Testing the Query Analyzer ---
User Question: 'feynman talking about algebra'
   Type of raw result: <class '__main__.TutorialSearch'>
{
  "content_search": "Feynman algebra",
  "title_search": "Feynman algebra"
}

User Question: 'lectures on math published before 2015'
   Type of raw result: <class '__main__.TutorialSearch'>
{
  "content_search": "lectures on math",
  "title_search": "math lectures",
  "earliest_publish_date": "2000-01-01",
  "latest_publish_date": "2015-01-01"
}

User Question: 'short videos under 3 minutes explaining variables like x and y'
   Type of raw result: <class '__main__.TutorialSearch'>
{
  "content_search": "short videos explaining variables like x and y",
  "title_search": "variables x y",
  "min_length_sec": 0,
  "max_length_sec": 180
}

User Question: 'richard feynman videos with over 1 million views'
   Type of raw result: <class '__main__.TutorialSearch'>
{
  "content_search": "richard feynman",
  "title_search": "richard feynman",
  "min_view